In [13]:
%pip install -r requirements.txt 2>/dev/null || echo "Some requirements failed to install – check above for details"

"Some requirements failed to install - check above for details"
Note: you may need to restart the kernel to use updated packages.


The system cannot find the path specified.


In [ ]:
import utils

<module 'utils' from 'c:\\Users\\edwin\\Documents\\Work\\DSO\\Learning\\Masters\\UofT\\Sem 2\\ECE1508\\Project\\ECE1508\\Code\\utils.py'>

In [43]:
%matplotlib QtAgg

results_root = 'models/seed_runs/'
results = utils.load_results(results_root)
results = utils.compute_stats(results)
utils.visualise_stats(results)

conformer_18M params: 18655950
conformer_18M test_loss mean: 0.30468032248621063
conformer_18M test_loss std: 0.0031746532925461847
conformer_18M test_cer mean: 0.08630550156037005
conformer_18M test_cer std: 0.0007216990474894926
conformer_18M test_wer mean: 0.25197844207286835
conformer_18M test_wer std: 0.0019186156027307982
conformer_5M params: 5185212
conformer_5M test_loss mean: 0.3425257382838706
conformer_5M test_loss std: 0.00302594173133738
conformer_5M test_cer mean: 0.11202574272950487
conformer_5M test_cer std: 0.0005265625127633033
conformer_5M test_wer mean: 0.3464446763197581
conformer_5M test_wer std: 0.002657588317618252
gru_lookahead params: 5217218
gru_lookahead test_loss mean: 0.31904172218912014
gru_lookahead test_loss std: 0.0037508813859317178
gru_lookahead test_cer mean: 0.10654962311188376
gru_lookahead test_cer std: 0.0007618832047001432
gru_lookahead test_wer mean: 0.33161549766858417
gru_lookahead test_wer std: 0.0024005957977173546
gru_no_lookahead_5 param

In [44]:
import matplotlib.pyplot as plt
plt.close('all')

## LM-augmented WER / CER Evaluation

Reload each saved `model_best.pth` per architecture and seed, run beam-search decoding with the KenLM 3-gram LM, and compute test WER / CER mean ± std across seeds.

In [ ]:
import torch
from pathlib import Path
from torchmetrics.text import CharErrorRate, WordErrorRate

from data_loader import download_ljspeech, download_lm, download_lexicon
from ljspeech import load_ljspeech
from decoder import Decoder
from deep_speech_2 import DeepSpeech2
from deep_speech_2_bidirectional import DeepSpeech2Bidirectional
from deep_speech_2_lstm import DeepSpeech2LSTM
from conformer import Conformer
from config import LM_ARPA_PATH, LEXICON_PATH, LM_WEIGHT, WORD_SCORE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Download LM + lexicon if not already present
download_lm(url="https://www.openslr.org/resources/11/3-gram.pruned.1e-7.arpa.gz", path="../data/lm")
download_lexicon(url="https://www.openslr.org/resources/11/librispeech-lexicon.txt", path="../data/lm")

# Download LJSpeech if not already present
LJSPEECH_URL = "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"
download_ljspeech(url=LJSPEECH_URL, path="../data/LJSpeech-1.1")

# Recreate the exact same 80/10/10 split used during training (seed=1508)
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 0
CACHE_DIR = "../data/cache/ljspeech"

lj_dataset = load_ljspeech("../data/LJSpeech-1.1/LJSpeech-1.1", cache_dir=CACHE_DIR)
n = len(lj_dataset)
lj_train_size = int(n * 0.8)
lj_val_size   = int(n * 0.1)
lj_test_size  = n - lj_train_size - lj_val_size
_, _, test_set = torch.utils.data.random_split(
    lj_dataset, [lj_train_size, lj_val_size, lj_test_size],
    generator=torch.Generator().manual_seed(1508),
)

test_loader = torch.utils.data.DataLoader(
    dataset=test_set,
    batch_size=EVAL_BATCH_SIZE,
    collate_fn=utils.collate_fn_eval,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
print(f"LJSpeech test set: {len(test_set)} utterances, {len(test_loader)} batches")

In [ ]:
MODEL_REGISTRY = {
    "gru_bidirectional_3": lambda: DeepSpeech2Bidirectional(
        conv_in_channels=1,
        in_feat_dim=80,
        GRU_hidden_size=256,
        GRU_depth=3,
        GRU_bidirectional=True,
    ),
    "gru_no_lookahead_5": lambda: DeepSpeech2Bidirectional(
        conv_in_channels=1,
        in_feat_dim=80,
        GRU_hidden_size=512,
        GRU_depth=5,
        GRU_bidirectional=False,
    ),
    "gru_no_lookahead_3": lambda: DeepSpeech2Bidirectional(
        conv_in_channels=1,
        in_feat_dim=80,
        GRU_hidden_size=512,
        GRU_depth=3,
        GRU_bidirectional=False,
    ),
    "lstm": lambda: DeepSpeech2LSTM(
        conv_in_channels=1,
        in_feat_dim=80,
        LSTM_hidden_size=512,
        LSTM_depth=3,
        LSTM_bidirectional=False,
        LSTM_dropout=0.3,
    ),
    "lstm_lookahead": lambda: DeepSpeech2LSTM(
        conv_in_channels=1,
        in_feat_dim=80,
        LSTM_hidden_size=512,
        LSTM_depth=3,
        LSTM_bidirectional=False,
        LSTM_dropout=0.3,
    ),
    "gru_lookahead": lambda: DeepSpeech2(
        conv_in_channels=1,
        in_feat_dim=80,
        GRU_hidden_size=512,
        GRU_depth=3,
    ),
    "conformer_5M": lambda: Conformer(
        conv_in_channels=1,
        enc_latent_dim=144,
        enc_feedforward_dim=576,
        enc_layers=10
    ),
}
print("Model registry:", list(MODEL_REGISTRY.keys()))

In [ ]:
import numpy as np
import pandas as pd

def _cache_model_outputs(model, test_loader, device):
    """Run one forward pass over the test set and cache (log_probs, out_lens) on CPU."""
    model.eval()
    cached = []
    with torch.no_grad():
        for batch in test_loader:
            specs = batch['padded_spectrograms'].to(device)
            seq_lens = batch['input_lengths']
            log_probs, out_lens = model(specs, seq_lens)
            cached.append((log_probs.cpu(), out_lens.cpu()))
    return cached


def compute_lm_stats(
    results_root: str,
    test_loader,
    model_registry: dict,
    device,
    lm_path: str = LM_ARPA_PATH,
    lexicon_path: str = LEXICON_PATH,
    lm_weight: float = LM_WEIGHT,
    word_score: float = WORD_SCORE,
):


    root = Path(results_root)
    lm_stats = {}

    for model_dir in sorted(root.iterdir()):
        if not model_dir.is_dir():
            continue
        arch_name = model_dir.name
        if arch_name not in model_registry:
            print(f"[SKIP] '{arch_name}' not in MODEL_REGISTRY")
            continue

        # Determine the best seed from seed_results.csv
        csv_path = model_dir / "seed_results.csv"
        if csv_path.exists():
            seed_df = pd.read_csv(csv_path)
            best_seed = int(seed_df.loc[seed_df['best_val_wer'].idxmin(), 'seed'])
        else:
            best_seed = None
            print(f"  [WARN] seed_results.csv not found in {model_dir}, best seed unknown")

        seed_wers, seed_cers = [], []
        best_seed_wer, best_seed_cer = None, None

        seed_dirs = sorted(d for d in model_dir.iterdir()
                           if d.is_dir() and d.name.startswith("seed_"))

        for seed_dir in seed_dirs:
            ckpt_path = seed_dir / "model_best.pth"
            if not ckpt_path.exists():
                print(f"  [SKIP] {ckpt_path} not found")
                continue

            # Rebuild model and load weights
            model = model_registry[arch_name]().to(device)
            utils.load_model(model=model, device=device, filepath=str(ckpt_path))

            # Single forward pass over the test set (reused for beam decoding)
            cached_outputs = _cache_model_outputs(model, test_loader, device)

            # Build KenLM beam decoder
            decoder = Decoder(
                tokenizer=model.tokenizer,
                blank_token=model.tokenizer.pad_token,
                lexicon=lexicon_path,
                lm=lm_path,
                lm_weight=lm_weight,
                word_score=word_score,
            )

            wer_metric = WordErrorRate()
            cer_metric = CharErrorRate()

            for batch, (log_probs, out_lens) in zip(test_loader, cached_outputs):
                hyps = decoder.decode_beam(
                    specs=None, spec_lengths=None, model=model,
                    log_probs=log_probs, out_lens=out_lens,
                )
                refs = utils._decode_targets(
                    batch['packed_transcripts'],
                    batch['target_lengths'],
                    model.tokenizer,
                )
                wer_metric.update(hyps, refs)
                cer_metric.update(hyps, refs)

            seed_wer = wer_metric.compute().item()
            seed_cer = cer_metric.compute().item()
            seed_wers.append(seed_wer)
            seed_cers.append(seed_cer)

            # Extract seed number from directory name (e.g. "seed_1508" -> 1508)
            current_seed = int(seed_dir.name.split("_")[1])
            if current_seed == best_seed:
                best_seed_wer = seed_wer
                best_seed_cer = seed_cer

            print(f"  {arch_name} / {seed_dir.name}  WER={seed_wer:.4f}  CER={seed_cer:.4f}"
                  + (" <-- best seed" if current_seed == best_seed else ""))

        if seed_wers:
            lm_stats[arch_name] = {
                'seed_wers': seed_wers,
                'seed_cers': seed_cers,
                'wer_mean': float(np.mean(seed_wers)),
                'wer_std':  float(np.std(seed_wers, ddof=1)) if len(seed_wers) > 1 else 0.0,
                'cer_mean': float(np.mean(seed_cers)),
                'cer_std':  float(np.std(seed_cers, ddof=1)) if len(seed_cers) > 1 else 0.0,
                'best_seed': best_seed,
                'best_seed_wer': best_seed_wer,
                'best_seed_cer': best_seed_cer,
            }

    return lm_stats

In [ ]:
lm_stats = compute_lm_stats(
    results_root=results_root,
    test_loader=test_loader,
    model_registry=MODEL_REGISTRY,
    device=device,
)

In [ ]:
print(f"\n{'Architecture':<25} {'WER (LM) mean':>14} {'WER (LM) std':>13} {'CER (LM) mean':>14} {'CER (LM) std':>13} {'Best seed WER':>14} {'Best seed CER':>14}")
print("-" * 112)
for arch, s in sorted(lm_stats.items()):
    bw = f"{s['best_seed_wer']:.4f}" if s['best_seed_wer'] is not None else "N/A"
    bc = f"{s['best_seed_cer']:.4f}" if s['best_seed_cer'] is not None else "N/A"
    print(f"{arch:<25} {s['wer_mean']:>14.4f} {s['wer_std']:>13.4f} {s['cer_mean']:>14.4f} {s['cer_std']:>13.4f} {bw:>14} {bc:>14}")

# Side-by-side comparison with greedy (no-LM) stats already in `results`
print(f"\n{'Architecture':<25} {'WER (greedy)':>13} {'WER (LM)':>10} {'WER (LM best)':>14} {'CER (greedy)':>13} {'CER (LM)':>10} {'CER (LM best)':>14}")
print("-" * 103)
for arch in sorted(lm_stats.keys()):
    greedy = results.get(arch)
    lm     = lm_stats[arch]
    bw = f"{lm['best_seed_wer']:.4f}" if lm['best_seed_wer'] is not None else "N/A"
    bc = f"{lm['best_seed_cer']:.4f}" if lm['best_seed_cer'] is not None else "N/A"
    if greedy is not None:
        gw = greedy.stats['test_wer'].mean
        gc = greedy.stats['test_cer'].mean
        print(f"{arch:<25} {gw:>13.4f} {lm['wer_mean']:>10.4f} {bw:>14} {gc:>13.4f} {lm['cer_mean']:>10.4f} {bc:>14}")
    else:
        print(f"{arch:<25} {'N/A':>13} {lm['wer_mean']:>10.4f} {bw:>14} {'N/A':>13} {lm['cer_mean']:>10.4f} {bc:>14}")